In [3]:
from IPython.core import page
from playwright.async_api import async_playwright
!pip install playwright
!playwright install


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from playwright.async_api import async_playwright

async def save_naver_session():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()

        page = await context.new_page()
        print("네이버 로그인 페이지로 이동합니다.")
        await page.goto("https://nid.naver.com/nidlogin.login")

        print("⚠여기서 직접 로그인하세요.")
        print("로그인 완료 후, 네이버 메인 화면이 보이면 아래 Enter를 눌러주세요.")

        input("로그인 완료 후 Enter 키를 누르세요 → ")

        print("세션 저장 중…")
        await context.storage_state(path="naver_state.json")
        print("세션 저장 완료! 이제 자동 업로드 가능!")

        await browser.close()

In [13]:
await save_naver_session()

네이버 로그인 페이지로 이동합니다.
⚠여기서 직접 로그인하세요.
로그인 완료 후, 네이버 메인 화면이 보이면 아래 Enter를 눌러주세요.
세션 저장 중…
세션 저장 완료! 이제 자동 업로드 가능!


In [14]:
from playwright.async_api import async_playwright
import asyncio

BLOG_ID = "rlarjsnd"


async def test_write_basic(title, content):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="naver_state.json")
        page = await context.new_page()

        print(" 글쓰기 페이지로 이동중…")
        await page.goto(f"https://blog.naver.com/{BLOG_ID}?Redirect=Write")
        await asyncio.sleep(1)

        # mainFrame 로드
        print(" mainFrame 로딩중…")
        await page.wait_for_selector("iframe[name='mainFrame']")
        frame = page.frame(name="mainFrame")
        await asyncio.sleep(0.3)

        # 기존 작성 중 팝업 닫기
        print(" 기존 작성 팝업 확인중…")

        try:
            # 팝업 dim이 먼저 나타남 → dim 이 나타나야 팝업 버튼도 활성화됨
            await frame.wait_for_selector("div.se-popup-dim", timeout=2000)
            print(" 팝업 발견 → 취소 클릭")

            await frame.click("button.se-popup-button.se-popup-button-cancel", force=True)
            await asyncio.sleep(0.8)

        except Exception:
            print("✔ 기존 작성 팝업 없음")

        await asyncio.sleep(0.4)

        # 2) 도움말 패널 닫기
        print("도움말 패널 확인중…")

        try:
            await frame.wait_for_selector(
                "button.se-help-panel-close-button",
                timeout=1800
            )
            print("도움말 패널 발견 → 닫기 클릭")

            await frame.click("button.se-help-panel-close-button", force=True)
            await asyncio.sleep(0.8)

        except Exception:
            print("도움말 패널 없음")

        await asyncio.sleep(0.3)

        # 3) 제목 입력

        print("제목 placeholder 찾는 중…")
        await frame.wait_for_selector("p.se-text-paragraph span.se-placeholder")

        print("제목 클릭!")
        await frame.click("p.se-text-paragraph span.se-placeholder")

        print("제목 입력중…")
        await frame.type("p.se-text-paragraph", title)
        print("제목 입력 SUCCESS!")

        # 4) 본문 입력

        print(" 본문 placeholder 찾는 중…")

        await frame.wait_for_selector(
            "div.se-module-text p.se-text-paragraph span.se-placeholder",
            timeout=5000
        )

        print(" 본문 클릭!")
        await frame.click("div.se-module-text p.se-text-paragraph span.se-placeholder")

        print(" 본문 입력중…")
        await frame.type("div.se-module-text p.se-text-paragraph", content)
        print(" 본문 입력 SUCCESS!")

        # 발행 버튼 1단계
        print(" 발행 버튼(1단계) 클릭 준비중…")

        try:
            await frame.wait_for_selector("button.publish_btn__m9KHH", timeout=3000)
            print(" 발행 버튼 발견 → 클릭중…")

            await frame.click("button.publish_btn__m9KHH", force=True)
            print(" 발행 버튼 1단계 클릭 성공!")

        except:
            print(" 발행 버튼을 찾지 못함… HTML 구조 확인 필요!")

        # 발행 버튼 2단계

        print(" 최종 발행(2단계) 버튼 확인중…")

        await frame.wait_for_selector("button[data-testid='seOnePublishBtn']", timeout=5000)
        await frame.click("button[data-testid='seOnePublishBtn']")

        print(" 게시물 발행 완료!")

        await asyncio.sleep(2)
        await browser.close()


# ------------------------------
# 실행 예시
# ------------------------------
await test_write_basic(
    title=" AURA 자동 업로드 제목 테스트!",
    content="자동 업로드 본문 내용 테스트입니다 "
)

 글쓰기 페이지로 이동중…
 mainFrame 로딩중…
 기존 작성 팝업 확인중…
✔ 기존 작성 팝업 없음
도움말 패널 확인중…
도움말 패널 발견 → 닫기 클릭
제목 placeholder 찾는 중…
제목 클릭!
제목 입력중…
제목 입력 SUCCESS!
 본문 placeholder 찾는 중…
 본문 클릭!
 본문 입력중…
 본문 입력 SUCCESS!
 발행 버튼(1단계) 클릭 준비중…
 발행 버튼 발견 → 클릭중…
 발행 버튼 1단계 클릭 성공!
 최종 발행(2단계) 버튼 확인중…
 게시물 발행 완료!
